# Fast flux readout (simple)

One readout at a fixed delay after the **rising edge** of a fixed fast flux pulse. Nothing is swept:
change the settings and run **Measure** again for another point. The same experiment as
`qick_tprocv2_experiments/027_fast_flux_readout_expt.py`, as a notebook.

```
Trigger   t = 0                               RFSoC pin HDAWG_TRIGGER_PIN
Flux      t = TRIG_OFFSET                     length FF_LENGTH, gain FF_GAIN
Readout   t = TRIG_OFFSET + READOUT_DELAY     inside the flux pulse if READOUT_DELAY < FF_LENGTH
Qubit     t = TRIG_OFFSET + Q_DELAY           commented out: see QUBIT PULSE in the helpers
```

After the flux pulse the HDAWG plays the same pulse inverted (compensation), `FF_COMPENSATION_WAIT` later.

**Before you start**
- The readout settings (frequency, gain, lengths, channels) and the qubit come from
  `qick_tprocv2_experiments/system_config.json`. The flux and timing settings are in the cell below.
- Each measurement is saved to `DATA_PATH` as `fast_flux_readout_Q<n>_<date>.h5`.

**Safety rules**
- `|FF_GAIN|` above `MAX_FF_GAIN` is refused, and the gain is read back from the HDAWG.
- The measurement refuses to run if the HDAWG would still be playing when the next trigger comes.
- After every measurement, also after an error or an interrupt, the flux gain goes to 0, every
  AWG core stops and the flux output switches off.

## Settings

In [ ]:
# ---- flux pulse and timing (us) ----
FF_LENGTH = 10.0            # flux pulse length (rounded to 16 HDAWG samples; the rounded length is saved)
FF_GAIN = -0.3              # flux amplitude, fraction of the HDAWG output range
READOUT_DELAY = 5.0         # rising edge of the flux pulse -> start of the readout
# Q_DELAY = 1.0             # rising edge of the flux pulse -> qubit pulse (QUBIT PULSE option)
TRIG_OFFSET = 0.02          # RFSoC trigger -> rising edge of the flux pulse
FF_COMPENSATION_WAIT = 15.0 # end of the flux pulse -> compensation pulse
MAX_FF_GAIN = 0.5           # |FF_GAIN| above this is refused

# ---- averaging ----
REPS = 1000
PY_AVG = 10
RELAX_DELAY = 150           # us between shots

# ---- HDAWG ----
HDAWG_ID = "devxxxx"
HDAWG_HOST, HDAWG_PORT = "xxx.xxx.xxx.xxx", "xxxx"    # LabOne data server
HDAWG_CHANNEL = 0           # front panel output counted from 0 (output 1 -> 0)
HDAWG_TRIGGER_PIN = 1       # RFSoC trigger pin wired to the HDAWG trigger input
HDAWG_SAMPLE_RATE = 2.4e9   # set on the instrument and read back
HDAWG_OUTPUT_RANGE = 1.0    # V; flux voltage = FF_GAIN x HDAWG_OUTPUT_RANGE

# ---- where the QICK experiment code and system_config.json are ----
EXPT_DIR = "qick_tprocv2_experiments"

## Helpers

Collapsed: click the cells to expand them. Run them once after starting the kernel.

In [ ]:
# Helpers, part 1: the HDAWG, one function per setup step (the same as hdawg_helpers.py).
#   connect_hdawg -> reset_and_configure -> set_sample_clock -> load_flux_pulse
#   -> set_flux_gain -> start_flux -> (RFSoC program) -> stop_flux
# Channel numbering: output counted from 0; in the 4x2 grouping AWG core n drives outputs 2n, 2n + 1.
# Only core 0 / output 1 has been used on the instrument so far.
import time
import numpy as np
from zhinst.toolkit import Session

GRANULARITY = 16    # HDAWG waveforms are a whole number of 16-sample blocks


def connect_hdawg(device_id, host, port):
    """Connect to the HDAWG through the LabOne data server; returns the device."""
    session = Session(host, port)
    print("HDAWG: devices visible:", session.devices.visible())
    device = session.connect_device(device_id)
    print("HDAWG: connected to", session.devices.connected())
    return device


def reset_and_configure(device, output_range=1.0, grouping=0):
    """
    Known state: factory reset, channel grouping (0 = 4x2), output range on every
    output, every AWG core off. The reset also resets the sample clock to 2.4 GS/s,
    so call set_sample_clock afterwards.
    """
    device.factory_reset()
    n_outputs = len(device.sigouts)
    with device.set_transaction():                 # one network round trip
        device.system.awg.channelgrouping(grouping)
        for ch in range(n_outputs):
            device.sigouts[ch].range(output_range)
    for core in range(len(device.awgs)):
        device.awgs[core].enable(0)
    print(f"HDAWG: reset, grouping {grouping}, output range {output_range} V, all cores off")


def set_sample_clock(device, rate):
    """Set the sample clock (<= 2.4e9) and check what the instrument reports; stops if it differs."""
    if rate > 2.4e9:
        raise ValueError(f"sample clock {rate} Hz is above the HDAWG maximum of 2.4 GS/s")
    device.system.clocks.sampleclock.freq(rate)
    device.system.clocks.sampleclock.status.wait_for_state_change(0, timeout=30)   # 0 = locked
    actual = device.system.clocks.sampleclock.freq()
    print(f"HDAWG: sample clock {actual:.3f} Hz (asked {rate:.3f} Hz)")
    if abs(actual - rate) > 0.5:
        raise RuntimeError(f"the HDAWG runs at {actual} Hz instead of {rate} Hz")
    return actual


def flux_samples(length_us, rate):
    """Samples in a flux pulse of length_us, rounded to 16 (6.67 ns at 2.4 GS/s)."""
    return GRANULARITY * int(round(length_us * 1e-6 * rate / GRANULARITY))


def flux_pulse_seqc(n_samples, compensation_wait_us, rate, local_channel=0):
    """SeqC: on every trigger a full-scale square pulse, then the same pulse inverted. The gain sets the amplitude."""
    wait_cycles = int(round(compensation_wait_us * 1e-6 * rate / 8))    # sequencer cycles
    index = local_channel + 1                                          # playWave counts outputs from 1
    return f"""
    // sample clock {rate:.0f} Hz, sequencer clock = sample clock / 8
    const pulse_samples = {n_samples};    // {n_samples / rate * 1e6:.6f} us
    const wait_cycles = {wait_cycles};    // {compensation_wait_us} us, in sequencer cycles

    wave w_rect = 1.0*rect(pulse_samples, 1);   // full scale; the output gain sets the amplitude

    while (true) {{
        waitDigTrigger(1);                          // RFSoC trigger
        playWave({index}, w_rect);                  // flux pulse
        wait(wait_cycles + pulse_samples / 8);      // pulse length plus the compensation wait
        playWave({index}, -1*w_rect);               // compensation pulse
    }}
    """


def load_flux_pulse(device, hdawg_channel, n_samples, compensation_wait_us, rate):
    """Compile and upload the flux pulse for one output (gain 0, core stopped) and switch the output on."""
    core_index, local = divmod(hdawg_channel, 2)        # two outputs per core
    awg_core = device.awgs[core_index]
    gain_node = awg_core.outputs[local].gains[local]    # /awgs/<core>/outputs/<local>/gains/<local>
    awg_core.enable(0)
    gain_node(0.0)                                      # silent while loading
    awg_core.load_sequencer_program(flux_pulse_seqc(n_samples, compensation_wait_us, rate, local))
    device.sigouts[hdawg_channel].on(True)
    print(f"HDAWG: flux pulse loaded on output {hdawg_channel + 1} (core {core_index}), "
          f"{n_samples} samples = {n_samples / rate * 1e6:.6f} us")
    return dict(awg_core=awg_core, gain_node=gain_node, channel=hdawg_channel)


def set_flux_gain(flux, gain, max_gain):
    """Set the flux amplitude (output gain) and read it back; refuses |gain| > max_gain."""
    if abs(gain) > max_gain + 1e-12:
        raise ValueError(f"flux gain {gain} is above the limit of {max_gain}")
    flux['gain_node'](gain)
    actual = float(flux['gain_node']())
    if abs(actual - gain) > 1e-3:
        raise RuntimeError(f"asked the HDAWG for gain {gain}, it reports {actual}")
    print(f"HDAWG: flux gain {actual:+.4f}")
    return actual


def start_flux(flux, settle_s=0.5):
    """Start the core: it plays the flux pulse on every RFSoC trigger from now on."""
    flux['awg_core'].single(True)
    flux['awg_core'].enable(1)
    time.sleep(settle_s)


def stop_flux(device, flux=None):
    """Gain to 0, every AWG core off, flux output off. Safe to call after an error."""
    if flux is not None:
        flux['gain_node'](0.0)
    for core in range(len(device.awgs)):
        device.awgs[core].enable(0)
    if flux is not None:
        device.sigouts[flux['channel']].on(False)
    print("HDAWG: flux off, all cores stopped")


def check_trigger_spacing(ff_length_us, compensation_wait_us, trig_offset_us, shot_period_us):
    """Refuse to run if the HDAWG (pulse, wait, compensation) is still busy when the next trigger comes."""
    busy = trig_offset_us + 2 * ff_length_us + compensation_wait_us
    if busy + 1 > shot_period_us:
        raise ValueError(f"the HDAWG needs {busy:.2f} us per trigger (pulse, {compensation_wait_us} us wait, "
                         f"compensation) but triggers come every {shot_period_us:.2f} us. "
                         "Increase RELAX_DELAY or shorten FF_LENGTH.")

In [ ]:
# Helpers, part 2: the QICK program, one measurement, and saving.
import os
import sys
import json
import datetime

_here = os.path.abspath(EXPT_DIR)
if os.path.basename(os.getcwd()) != os.path.basename(_here):
    os.chdir(_here)                          # system_config.json is read from the working directory
if _here not in sys.path:
    sys.path.insert(0, _here)

from qick import *
from qick.asm_v2 import AveragerProgramV2
from malab import SlabFile, get_next_filename
from qick_setup_funcs import *           # initialize_configs, initialize_ro_chs, MUX, QUBIT_INDEX, DATA_PATH, ...


def build_config():
    """The fast_flux_readout config from system_config.json (readout, qubit), with the settings above on top."""
    config, expt_name = initialize_configs("fast_flux_readout")
    config.update(ff_length=FF_LENGTH, ff_gain=FF_GAIN, readout_delay=READOUT_DELAY,
                  trig_offset=TRIG_OFFSET, ff_compensation_wait=FF_COMPENSATION_WAIT, max_ff_gain=MAX_FF_GAIN,
                  reps=REPS, py_avg=PY_AVG, relax_delay=RELAX_DELAY,
                  hdawg_channel=HDAWG_CHANNEL, hdawg_trigger_pin=HDAWG_TRIGGER_PIN,
                  hdawg_sample_rate=HDAWG_SAMPLE_RATE, hdawg_output_range=HDAWG_OUTPUT_RANGE)
    # config['q_delay'] = Q_DELAY                    # QUBIT PULSE option
    return config, expt_name


class FastFluxReadoutProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        # readout channels and pulses for DAC and ADC
        initialize_ro_chs(self, cfg, suffix='_ge')

        # ----- QUBIT PULSE (option): uncomment to declare the qubit pulse ----- #
        # declare_gen_ch(self, cfg, cfg['qubit_ch'], usage='qubit', suffix='_ge')
        # declare_pulse(self, cfg, cfg['qubit_ch'], usage='qubit',
        #               pulse_style='gauss', pulse_name='qubit_pulse', suffix='_ge')

    def _body(self, cfg):
        t_ro = cfg['trig_offset'] + cfg['readout_delay']    # readout start, from the trigger

        # trigger the HDAWG: the flux pulse starts trig_offset later
        self.trigger(pins=[cfg['hdawg_trigger_pin']], t=0)

        # ----- QUBIT PULSE (option): uncomment to play it q_delay after the rising edge ----- #
        # self.pulse(ch=cfg['qubit_ch'], name='qubit_pulse', t=cfg['trig_offset'] + cfg['q_delay'])

        # readout readout_delay after the rising edge of the flux pulse
        if MUX == 'True':
            self.pulse(ch=cfg['mux_ch'], name='res_pulse', t=t_ro)
            self.trigger(ros=cfg['mux_ro_chs'], pins=[0], t=cfg['trig_time'] + t_ro)
        else:
            self.send_readoutconfig(ch=cfg['ro_ch'], name='myro', t=0)
            self.pulse(ch=cfg['res_ch'], name='res_pulse', t=t_ro)
            self.trigger(ros=[cfg['ro_ch']], pins=[0], t=cfg['trig_time'] + t_ro)


def measure(soc, soccfg, device, config):
    """
    One measurement: load the flux pulse, set the gain, run the RFSoC program, and turn
    the flux off again (also after an error or an interrupt). Returns a dict of results.
    """
    rate = config['hdawg_sample_rate']
    n_samples = flux_samples(config['ff_length'], rate)
    ff_length = n_samples / rate * 1e6                  # the length the HDAWG plays, us
    if config['readout_delay'] < 0:
        raise ValueError("READOUT_DELAY must be >= 0: the readout starts after the rising edge")
    check_trigger_spacing(ff_length, config['ff_compensation_wait'], config['trig_offset'],
                          config['trig_offset'] + config['readout_delay'] + config['ro_length'] + config['relax_delay'])
    where = "inside" if config['readout_delay'] < ff_length else "after"
    print(f"flux pulse {ff_length:.6f} us, readout {config['readout_delay']} us after the rising edge ({where} the pulse)")

    start = time.time()
    flux = load_flux_pulse(device, config['hdawg_channel'], n_samples, config['ff_compensation_wait'], rate)
    try:
        ff_gain = set_flux_gain(flux, config['ff_gain'], config['max_ff_gain'])
        start_flux(flux)
        prog = FastFluxReadoutProgram(soccfg, reps=config['reps'], final_delay=config['relax_delay'], cfg=config)
        iq_list = prog.acquire(soc, soft_avgs=config['py_avg'], progress=True)
    finally:
        stop_flux(device, flux)

    ro_index = QUBIT_INDEX - 1 if MUX == 'True' else 0      # no loops: one averaged (I, Q) per readout channel
    I, Q = (float(v) for v in np.ravel(iq_list[ro_index])[:2])
    result = dict(I=I, Q=Q, amp=abs(I + 1j * Q), phase=float(np.angle(I + 1j * Q)),
                  ff_length=ff_length, ff_gain=ff_gain, readout_delay=config['readout_delay'],
                  trig_offset=config['trig_offset'], expt_time=time.time() - start)
    print(f"I = {I:.4f}, Q = {Q:.4f}, |IQ| = {result['amp']:.4f}, phase = {result['phase']:.4f} rad")
    return result


def save(result, config, expt_name, data_path):
    """Save one measurement as <n>_fast_flux_readout_Q<qubit>_<date>.h5 in data_path; returns the filename."""
    exp_name = expt_name + '_Q' + str(config['qubit_index']) + '_' + str(datetime.date.today())
    fname = get_next_filename(data_path, exp_name, suffix='.h5')
    with SlabFile(os.path.join(data_path, fname), 'a') as f:
        f.append('expt_time', [int(result['expt_time'])])
        f.append('avgi', [result['I']])
        f.append('avgq', [result['Q']])
        f.append('amps', [result['amp']])
        f.append('ff_length', [result['ff_length']])        # the length played
        f.append('ff_gain', [result['ff_gain']])            # as read back from the HDAWG
        f.append('ff_volts', [result['ff_gain'] * config['hdawg_output_range']])
        f.append('readout_delay', [result['readout_delay']])
        f.append('trig_offset', [result['trig_offset']])
        # f.append('q_delay', [config['q_delay']])          # QUBIT PULSE option
        f.attrs['config'] = json.dumps(config)
    print(f"saved {fname} in {data_path}")
    return fname

In [ ]:
# Helpers, part 3: analysis of saved measurements.
import glob
import h5py
import matplotlib.pyplot as plt


def load_measurements(data_path, files=None, date=None):
    """
    Saved fast_flux_readout measurements as a list of dicts, sorted by readout_delay.

    files: list of filenames in data_path; None: every fast_flux_readout file of `date`
    date: 'YYYY-MM-DD'; None: today
    """
    if files is None:
        date = date or str(datetime.date.today())
        files = sorted(os.path.basename(p) for p in glob.glob(os.path.join(data_path, f"*_fast_flux_readout*_{date}.h5")))
    rows = []
    for fname in files:
        with h5py.File(os.path.join(data_path, fname), 'r') as h:
            first = lambda k: float(np.ravel(h[k])[0]) if k in h else np.nan
            I, Q = first('avgi'), first('avgq')
            rows.append(dict(file=fname, I=I, Q=Q, amp=abs(I + 1j * Q), phase=float(np.angle(I + 1j * Q)),
                             readout_delay=first('readout_delay'), ff_length=first('ff_length'),
                             ff_gain=first('ff_gain'), q_delay=first('q_delay')))
    rows.sort(key=lambda r: (r['ff_gain'], r['ff_length'], r['readout_delay']))
    return rows


def show_measurements(rows):
    """A table of the measurements."""
    print(f"{'file':42} {'gain':>7} {'length':>8} {'ro delay':>9} {'I':>9} {'Q':>9} {'|IQ|':>9} {'phase':>7}")
    for r in rows:
        print(f"{r['file']:42} {r['ff_gain']:>+7.3f} {r['ff_length']:>8.3f} {r['readout_delay']:>9.3f} "
              f"{r['I']:>9.4f} {r['Q']:>9.4f} {r['amp']:>9.4f} {r['phase']:>7.3f}")


def plot_measurements(rows, data_path=""):
    """
    Left: every measurement in the IQ plane, labelled with its readout delay.
    Right: |IQ| and phase against readout delay, the flux pulse end marked.
    One colour per (flux gain, flux length).
    """
    groups = {}
    for r in rows:
        groups.setdefault((round(r['ff_gain'], 4), round(r['ff_length'], 4)), []).append(r)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 6))
    a3 = a2.twinx()
    for k, ((gain, length), rs) in enumerate(groups.items()):
        c, lab = f"C{k}", f"gain {gain:+.3g}, pulse {length:.4g} us"
        a1.plot([r['I'] for r in rs], [r['Q'] for r in rs], 'o', color=c, label=lab)
        for r in rs:
            a1.annotate(f"{r['readout_delay']:g}", (r['I'], r['Q']), xytext=(4, 4), textcoords='offset points',
                        fontsize=8, color=c)
        d = [r['readout_delay'] for r in rs]
        a2.plot(d, [r['amp'] for r in rs], 'o-', color=c, label=lab)
        a3.plot(d, [r['phase'] for r in rs], 's--', color=c, mfc='none', alpha=0.6)
        a2.axvline(length, color=c, ls=':', lw=1)
    a1.set_xlabel("I (a.u.)"); a1.set_ylabel("Q (a.u.)"); a1.set_aspect('equal', adjustable='datalim')
    a1.set_title("IQ plane, labels: readout delay (us)")
    a1.legend(fontsize=8)
    a2.set_xlabel("Readout delay from the flux rising edge (us)")
    a2.set_ylabel("|IQ| (a.u.), circles"); a3.set_ylabel("phase (rad), squares")
    a2.set_title("dotted: end of the flux pulse")
    fig.suptitle(f"Fast flux readout, {len(rows)} measurements")
    fig.text(0.5, 0.005, data_path, ha='center', fontsize=7)
    fig.tight_layout(rect=(0, 0.02, 1, 1))
    plt.show()
    return fig

## Connect

Connects to the RFSoC (`rfsoc_connect`) and the HDAWG, resets the HDAWG and sets its sample clock.
No flux pulse is played here.

In [ ]:
from rfsoc_connect import soc, soccfg
device = connect_hdawg(HDAWG_ID, HDAWG_HOST, HDAWG_PORT)
reset_and_configure(device, output_range=HDAWG_OUTPUT_RANGE)
set_sample_clock(device, HDAWG_SAMPLE_RATE)

## Measure

Rerun **Settings** after changing it, then this cell. One measurement, saved to `DATA_PATH`.
Interrupt the kernel to stop: the flux goes off.

In [ ]:
config, expt_name = build_config()
result = measure(soc, soccfg, device, config)
last_file = save(result, config, expt_name, DATA_PATH)

## Flux off

The measurement does this itself; here in case you need it by hand.

In [ ]:
stop_flux(device)

## Analysis

Every fast_flux_readout measurement of a day (or a list of files), as a table and plotted:
the IQ plane, and |IQ| and phase against readout delay, one colour per flux gain and length.

In [ ]:
ANALYSIS_FILES = None      # None: every fast_flux_readout file of ANALYSIS_DATE; or a list of filenames
ANALYSIS_DATE = None       # 'YYYY-MM-DD'; None: today

rows = load_measurements(DATA_PATH, ANALYSIS_FILES, ANALYSIS_DATE)
show_measurements(rows)
if rows:
    fig = plot_measurements(rows, DATA_PATH)